In [2]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader

C:\Users\User\AppData\Local\Temp\ipykernel_20900\3894764843.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [3]:
# Read pdf file 
# File path
file_path = "Código do trabalho.pdf"
print(f"⏳ Loading {file_path}...")
# Loading the PDF file and splitting it into pages
loader = PyPDFLoader(file_path)
documents = loader.load_and_split()
print(f"✅ Success! O PDF foi lido e dividido em {len(documents)} páginas prontas para o sistema de busca.")

⏳ Loading Código do trabalho.pdf...
✅ Success! O PDF foi lido e dividido em 322 páginas prontas para o sistema de busca.


In [4]:
# Initialize the OllamaEmbeddings with a timeout to prevent slow responses or timeouts
embeddings_locais = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434",
    client_kwargs={
        "timeout": 120.0  # Protection against slow responses or timeouts
    }
)

# Save the embeddings to a Chroma vector store
# Create a Chroma vector store from the documents and embeddings
vectorstore = Chroma.from_documents(documents, embedding=embeddings_locais)

print("✅ Data saved!")

✅ Data saved!


In [5]:
# Configure the vector retriever to use the Chroma vector store
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Configurate the BM25Retriever to use the documents directly instead of the vectorstore
# Change the BM25Retriever to use the documents directly instead of the vectorstore
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 2

# Create a hybrid retriever function that combines both retrieval methods
def hybrid_retriever_func(query: str):
    # Execute both retrieval methods
    docs_vector = vector_retriever.invoke(query)
    docs_bm25 = bm25_retriever.invoke(query)
    
    # Merge the results, ensuring no duplicates based on page content
    seen = set()
    combined_docs = []
    for doc in docs_vector + docs_bm25:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            combined_docs.append(doc)
            
    return combined_docs

print("✅ System ready! You can now ask questions about the PDF content using the hybrid search function.")

✅ System ready! You can now ask questions about the PDF content using the hybrid search function.


In [6]:
# Test the hybrid retriever with a sample question
test_question = "Falame sobre trabalho infantil"
documents_found = hybrid_retriever_func(test_question)

# Show the results of the hybrid search
print(f"🔍 Question: '{test_question}'")
print(f"📄 The system found {len(documents_found)} relevant snippet(s):\n")

for idx, doc in enumerate(documents_found):
    print(f"Snippet {idx+1}: {doc.page_content}")

🔍 Question: 'Falame sobre trabalho infantil'
📄 The system found 4 relevant snippet(s):

Snippet 1: Documento atualizado a 28 de abril de 2023 
 
 
 
 
 
Artigo 252.º 
 
Falta para assistência a membro do agregado familiar 
 
1 – O trabalhador tem direito a faltar ao trabalho até 15 dias por ano para prestar assistência inadiável e 
imprescindível, em caso de doença ou acidente, a cônjuge o u pessoa que viva em união de facto ou 
economia comum com o trabalhador, parente ou afim na linha reta ascendente ou no 2.º grau da linha 
colateral. 
 
2 - O direito previsto no número anterior é ainda garantido ao trabalhador cuidador a quem seja 
reconhecido o estatuto de cuidador informal não principal, em caso de doença ou acidente da pessoa 
cuidada, nos termos definidos na legislação aplicável.1 
 
3 – Ao período de ausência previsto no n.º 1 acrescem 15 dias por ano, no caso de prestação de assistência 
inadiável e imprescindível a pessoa com deficiência ou doença crónica, que seja cônjuge o

In [7]:
from langchain_ollama import ChatOllama

In [8]:
print("⏳ Initializing the Qwen model in Ollama (make sure Ollama is running)...")

# Initialize the ChatOllama model with the specified parameters
llm_local = ChatOllama(
    model="qwen2.5:7b",
    temperature=0  # Keep the temperature at 0 for deterministic responses
)

print("✅ Model Qwen2.5:7b initialized and ready!")

⏳ Initializing the Qwen model in Ollama (make sure Ollama is running)...
✅ Model Qwen2.5:7b initialized and ready!


In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [12]:
# Create a prompt template that instructs the model to answer using only the provided context
# Only use the context provided and if the answer is not found, respond honestly that the information is not available.
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the user's question using ONLY the context provided below. If you don't know the answer based on the context, honestly say that you couldn't find the information.\n\Context:\n{context}"),
    ("human", "{question}")
])

# Function to handle user questions, retrieve relevant documents, and generate answers
def responder_pergunta(user_question: str):
    # Find relevant documents using the hybrid retriever
    docs_relevantes = hybrid_retriever_func(user_question)
    
    # Merge the content of the relevant documents into a single context string
    merged_context = "\n".join([doc.page_content for doc in docs_relevantes])
    
    # Prepare the prompt for the model using the merged context and the user's question
    formated_prompt = prompt_template.format_messages(
        context=merged_context,
        question=user_question
    )
    
    # Qwen will generate an answer based on the provided context and question
    answer = llm_local.invoke(formated_prompt)
    
    # Use the output parser to clean and format the answer before returning it
    parser = StrOutputParser()
    return parser.invoke(answer)

# --- Real Test ---
question = "Quais são os principais deveres do trabalhador"
print(f"🤔 Pergunta feita: {question}\n")

print("🤖 O Qwen está processando a resposta...")
final_answer = responder_pergunta(question)

print("-" * 50)
print(f"✨ Resposta da IA:\n{final_answer}")
print("-" * 50)

🤔 Pergunta feita: Quais são os principais deveres do trabalhador

🤖 O Qwen está processando a resposta...
--------------------------------------------------
✨ Resposta da IA:
Os principais deveres do trabalhador, conforme estabelecido no Artigo 128.º, incluem:

a) Respeitar e tratar o empregador, superiores hierárquicos, companheiros de trabalho e pessoas que se relacionem com a empresa com urbanidade e probidade;
b) Comparecer ao serviço com assiduidade e pontualidade;
c) Realizar o trabalho com zelo e diligência;
d) Participar de modo diligente em ações de formação profissional proporcionadas pelo empregador;
e) Cumprir as ordens e instruções do empregador respeitantes à execução ou disciplina do trabalho, bem como segurança e saúde no trabalho, desde que não sejam contrárias aos seus direitos ou garantias;
f) Guardar lealdade ao empregador, evitando concorrer com ele por conta própria ou alheia e não divulgando informações confidenciais da empresa;
g) Velar pela conservação e boa ut